# Train on Colab
Runtime > Change runtime type > GPU (T4) before running.

In [ ]:
# 1. Clone (or update) the repo
import os
REPO = 'https://github.com/ahmed-khaled04/bird_image_to_caption.git'
ROOT = '/content/bird_image_to_caption'
if not os.path.exists(ROOT):
    !git clone {REPO} {ROOT}
else:
    !git -C {ROOT} pull
%cd {ROOT}

In [ ]:
# 2. Dependencies (torch/torchvision are already on Colab, so they are skipped)
!grep -vi -E '^(torch|torchvision)$' requirements.txt > /tmp/req.txt
!pip install -q -r /tmp/req.txt
import torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# 3. Keep the trained weights on Drive so they survive the session.
# Only models/ is linked: outputs/ holds tracked files (e.g. tables/duplicate_pairs.csv, needed by `clean`).
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/bird_caption_generator'
os.makedirs(f'{DRIVE}/models', exist_ok=True)
!rm -rf {ROOT}/models
!ln -s {DRIVE}/models {ROOT}/models
# restore anything an earlier version of this cell deleted from the repo
!git -C {ROOT} checkout -- outputs
assert os.path.exists(f'{ROOT}/outputs/tables/duplicate_pairs.csv')

In [ ]:
# 4. Make `import src...` work and build the cleaned data (data/ is not in git)
import sys
sys.path.insert(0, ROOT)
!python -m src.cli clean --no-plots
assert os.path.exists('data/processed'), 'clean failed: read the error above'

In [ ]:
# 5. Quick check on a few images, then the real run. --num-workers 2 is fine on Colab.
!python -m src.cli transfer --limit 64 --num-workers 2
!python -m src.cli transfer --num-workers 2

In [ ]:
# Optional: call the code directly from Python instead of the CLI
from pathlib import Path
import yaml
from src.experiment import load_processed, train_run
data = load_processed(Path('data/processed'))
config = yaml.safe_load(Path('configs/transfer.yaml').read_text())
print([r['name'] for r in config['runs']])